# Train and Evaluate Geometry Model

Train and evaluate the PhaseGridUNet model.

**Workflow**

1. Set `PRESET = "smoke"` and run all sections to verify the training and evaluation pipeline.
2. Set `PRESET = "development"` and run all sections to train the development model.
3. Review the training history, evaluation metrics and prediction visualizations.

To inspect an existing training run, set `DATASET_NAME` and `RUN_NAME` accordingly and skip training.


## 1. Setup

Use the corresponding experiment's Python environment as the notebook kernel.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from grid_geometry.phase.train import train
from grid_geometry.phase.visualize import plot_history
from grid_geometry.dataset.loader import PhaseDataset
from grid_geometry.phase.runs import load_saved_run, prepare_training, project_directory, training_config
from grid_geometry.phase.reporting import recognition_counts, outcome_matrix, occupancy_matrix, fraction, plot_matrices
from grid_geometry.phase.evaluation_runs import evaluate_run
from grid_geometry.phase.diagnostics import visualize_predictions
from grid_geometry.notebook_display import show_table, show_mapping


## 2. Configuration

Set `DATASET_NAME` to select the input dataset folder and `RUN_NAME` to select the output run folder. Use `DEVICE` to select the CPU or an accelerator, or `auto` for automatic selection.


In [ ]:
PROJECT = project_directory(Path.cwd())
DATASETS = PROJECT / "datasets"
RUNS = PROJECT / "runs"

PRESET = "smoke"       # "smoke", "development"
DATASET_NAME = PRESET
RUN_NAME = PRESET
DEVICE = "auto"        # "auto", "cpu", "mps", "cuda"

presets = {
    "smoke": {
        "epochs": 2,
        "cosine_t_max": 2,
        "batch_size": 8,
    },
    "development": {
        "epochs": 50,
        "cosine_t_max": 50,
        "batch_size": 16,
    },
}
if PRESET not in presets:
    raise ValueError(f"Unknown preset: {PRESET}")
training = dict(
    workers=0,
    optimizer="adamw",
    learning_rate=0.001,
    weight_decay=0.0001,
    scheduler="cosine",  # "none", "step", "cosine"
    cosine_eta_min=1e-5,
    seed=42,
    device=DEVICE,
    amp=True,
    occupancy_weight=1.0,
    selection_metric="loss",
    patience=10,
    **presets[PRESET],
)
config = training_config(PROJECT, DATASET_NAME, RUN_NAME, training)
print(f"Preset: {PRESET}; model: PhaseGridUNet; input: 384 x 384; epochs: {training['epochs']}; batch: {training['batch_size']}")
print(f"Requested device: {DEVICE}; requested AMP: {training['amp']}")


## 3. Train model

Train PhaseGridUNet from random initialization to predict directional phase and occupancy. An existing run folder is protected from accidental overwrite.


In [ ]:
prepare_training(config)
history = train(config)


## 4. Load saved run

The following analysis uses the saved training configuration, history and checkpoints rather than assuming the current training settings were used.


In [ ]:
run = load_saved_run(RUNS, RUN_NAME)
inspect_dir, last_path, best_path = run.directory, run.last_path, run.best_path
state, best_state, saved_config = run.state, run.best_state, run.config
saved_root, saved_dataset_config = run.dataset_root, run.dataset_config
history, outcomes = run.history, run.outcomes
print(f"Run: {inspect_dir}")
print(f"Dataset: {saved_root}")
print(f"Last checkpoint: {last_path}")
print(f"Best checkpoint: {best_path}")


## 5. Analyze training history

Follow phase and occupancy loss, angular error, occupancy IoU and learning-rate progression across epochs. Compare training and validation trends to assess convergence and possible overfitting.


In [ ]:
fig = plot_history(history)
best_epoch = outcomes["best_epoch"]
if best_epoch is not None:
    fig.axes[0].axvline(best_epoch, color="black", linestyle=":", label=f"best epoch {best_epoch}")
    fig.axes[0].legend()
plt.show()
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for split in ("train", "validation"):
    for metric in ("phase_loss", "occupancy_loss", "phase_mae_radians_x", "phase_mae_radians_y", "iou"):
        points = [(row["epoch"], row[split]["metrics"]["all"].get(metric)) for row in history]
        points = [(x, y) for x, y in points if y is not None]
        if points:
            ax = axes[0] if metric.endswith("loss") else axes[2] if metric == "iou" else axes[1]
            ax.plot(*zip(*points), marker="o", markersize=3, label=f"{split} {metric}")
for ax, title in zip(axes, ("Loss components", "Phase angular error (radians)", "Occupancy IoU")):
    ax.set(title=title, xlabel="Epoch")
    ax.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=8))
    if ax.lines: ax.legend(fontsize=8)
axes[2].yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1, decimals=2))
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 3))
points = [(r["epoch"], r.get("learning_rate")) for r in history if r.get("learning_rate") is not None]
if points:
    ax.plot(*zip(*points), marker="o", markersize=3)
ax.set(title="Learning rate", xlabel="Epoch")
ax.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=8))
fig.tight_layout()
plt.show()


## 6. Analyze training performance

Epoch duration, throughput and the recorded environment describe the computational cost and execution context of the saved run. The reported effective device and AMP settings can be compared with the saved requested settings.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, title, series in (
    (axes[0], "Epoch duration (s)", [("epoch", lambda r: r.get("performance", {}).get("epoch_seconds"))]),
    (axes[1], "Throughput (samples/s)", [(s, lambda r, s=s: r[s].get("samples_per_second")) for s in ("train", "validation")]),
):
    for label, get in series:
        points = [(r["epoch"], get(r)) for r in history if get(r) is not None]
        if points: ax.plot(*zip(*points), marker="o", markersize=3, label=label)
    ax.set(title=title, xlabel="Epoch")
    ax.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=8))
    if ax.lines: ax.legend(fontsize=8)
    else: ax.text(.5, .5, "Unavailable", ha="center", transform=ax.transAxes)
fig.tight_layout()
plt.show()

env = state.get("environment") or {}
memory = state.get("device_memory") or {}
def available(value):
    return "—" if value is None else str(value)
print(f"Saved training environment: Python {available(env.get('python'))} | PyTorch {available(env.get('torch'))} | {available(env.get('platform'))}")
print(f"Device: {available(outcomes.get('device_effective'))} | AMP: {available(outcomes.get('amp_effective'))} | Workers: {available(saved_config.get('workers'))}")
for label, key in (("Peak allocated", "peak_allocated_bytes"), ("Peak reserved", "peak_reserved_bytes")):
    value = memory.get(key)
    print(f"CUDA {label.lower()}: {value / 2**20:.1f} MiB" if value is not None else f"CUDA {label.lower()}: —")
rates = outcomes.get("aggregate_samples_per_second") or {}
for split in ("train", "validation"):
    value = rates.get(split)
    print(f"{split.capitalize()} throughput: {value:.2f} samples/s" if value is not None else f"{split.capitalize()} throughput: —")


## 7. Evaluate model

Geometry precision measures the fraction of accepted recoveries that are correct; recall measures successful recovery among eligible samples. Local lattice recovery is reported separately from complete board recognition, which requires correct full-board geometry. Recognition outcome confusion matrices show acceptance and rejection decisions by crop type, while phase error and occupancy metrics diagnose network predictions. Compare validation and test results and subgroup charts for sensitivity to crop category, cell pitch and resampling.


In [ ]:
EVALUATION_LIMIT = None  # positive integer for a seeded debugging subset
EVALUATION_SEED = 42
reports = evaluate_run(run, device=DEVICE, limit=EVALUATION_LIMIT, seed=EVALUATION_SEED)
splits_to_compare = ("val", "test")
for split in splits_to_compare:
    total = len(PhaseDataset(saved_root, split, require_nonempty=True))
    print(f"{split}: evaluated {len(reports[split]['records'])} of {total} samples; subset seed {EVALUATION_SEED}")

def percent(value):
    return "Unavailable" if value is None else f"{value:.2%}"
def phase(value):
    return "Unavailable" if value is None else f"{100 * value / (2 * 3.141592653589793):.2f}%"
def number(value, suffix=""):
    return "Unavailable" if value is None else f"{value:.3f}{suffix}"

def table(title, rows):
    show_table(title, ("Metric", "Validation", "Test"),
               [(name, *(get(reports[s]) for s in splits_to_compare)) for name, get in rows])

table("1. Heatmap prediction", [
    ("Phase error X (% of pitch)", lambda r: phase(r["target_metrics"]["all"]["phase_mae_radians_x"])),
    ("Phase error Y (% of pitch)", lambda r: phase(r["target_metrics"]["all"]["phase_mae_radians_y"])),
    *((f"Occupancy {key}", lambda r, key=key: percent(r["target_metrics"]["all"][key])) for key in ("precision", "recall", "iou"))])
plot_matrices({"Validation": occupancy_matrix(reports["val"]), "Test": occupancy_matrix(reports["test"])},
              ("Board", "Background"), ("Board", "Background"), "Occupancy outcomes (supervised pixels)")

counts = {s: recognition_counts(reports[s]["records"]) for s in splits_to_compare}
show_table("2. Local lattice recognition", ("Metric", "Validation", "Test"),
           [(name, *(fraction(counts[s][name]) for s in splits_to_compare))
            for name in ("Local lattice precision", "Local lattice recall")])
table("Local lattice decisions", [
    ("Correct hypotheses / positives", lambda r: fraction((r["geometry"]["local_geometry_recovery"]["lattice"]["correct_hypotheses"], r["geometry"]["positives"]))),
    ("Correct accepted / positives", lambda r: fraction((r["geometry"]["local_geometry_recovery"]["lattice"]["correct_accepted"], r["geometry"]["positives"]))),
    *((f"{category} local false acceptance", lambda r, category=category: percent(r["geometry"]["local_geometry_recovery"]["negative_by_category"][category]["false_acceptance_rate"])) for category in ("negative", "hard_negative"))])
table("Local lattice errors (all positive hypotheses)", [
    (label, lambda r, key=key: number(r["geometry"][key], " px")) for label, key in (
        ("Pitch X", "mean_pitch_error_x_px"), ("Pitch Y", "mean_pitch_error_y_px"),
        ("Origin alignment X", "mean_alignment_error_x_px"), ("Origin alignment Y", "mean_alignment_error_y_px"))])

show_table("3. Local geometry recovery", ("Metric", "Validation", "Test"),
           [(name, *(fraction(counts[s][name]) for s in splits_to_compare))
            for name in ("Local geometry precision", "Local geometry recall")])
def local(r): return r["geometry"]["local_geometry_recovery"]
table("Local geometry and equivalent boundaries", [
    ("Correct geometry hypotheses / positives", lambda r: fraction((local(r)["geometry"]["correct_hypotheses"], r["geometry"]["positives"]))),
    ("Correct accepted geometry / positives", lambda r: fraction((local(r)["geometry"]["correct_accepted"], r["geometry"]["positives"]))),
    *((f"{label} boundary {metric}", lambda r, stage=stage, metric=metric: percent(local(r)[stage][metric]))
      for label, stage in (("Hypothesis", "hypothesis_boundaries"), ("Accepted", "accepted_boundaries"))
      for metric in ("precision", "recall", "line_selection_rate")),
    ("Hypothesis boundary error (px; all available)", lambda r: number(local(r)["hypothesis_boundaries"]["mean_error_px"], " px")),
    ("Hypothesis boundary error (cells; all available)", lambda r: number(local(r)["hypothesis_boundaries"]["mean_error_cells"], " cells"))])
for breakdown, keys in (("by_category", ("complete", "interior", "side", "corner")),
                        ("by_direction", ("left", "top", "right", "bottom"))):
    def values(split, key):
        item = local(reports[split])[breakdown][key]
        boundary = item["accepted_boundaries"] if breakdown == "by_category" else item["accepted"]
        return fraction((boundary["correct"], boundary["expected"]))
    show_table(f"Accepted boundary recall by {breakdown[3:]}", ("Group", "Validation", "Test"),
               [(key, *(values(s, key) for s in splits_to_compare)) for key in keys])
print("Equivalent boundaries describe the local known extent, including midpoint alternatives; they do not assert physical board termination. Boundary metrics use positive samples only; rejected sides count as misses.")

show_table("4. Global complete-board recognition (single crop)", ("Metric", "Validation", "Test"),
           [(name, *(fraction(counts[s][name]) for s in splits_to_compare)) for name in (
               "Complete board precision", "Complete board recall", "Partial board false acceptance rate", "Negative false acceptance rate")])
table("Physical board geometry (complete samples with global geometry)", [
    ("Rows correct", lambda r: fraction((r["global_geometry_single_crop"]["complete_rows_correct"], r["global_geometry_single_crop"]["complete_with_geometry"]))),
    ("Columns correct", lambda r: fraction((r["global_geometry_single_crop"]["complete_columns_correct"], r["global_geometry_single_crop"]["complete_with_geometry"])))])
plot_matrices({"Validation": outcome_matrix(reports["val"]["records"]), "Test": outcome_matrix(reports["test"]["records"])},
              ("Complete board", "Partial board", "Negative"), ("Complete board", "Local lattice only", "Rejected"),
              "Single-crop recognition outcomes")
print("Local lattice only means the global input crop passed local validation but global complete-board recognition rejected it.")


In [ ]:
# Join evaluated records to saved sample metadata; no inference or decoding is repeated.
def sample_metadata(split, sample_id):
    return json.loads((saved_root / split / f"{sample_id}.json").read_text())
metadata_by_split = {split: {r["sample_id"]: sample_metadata(split, r["sample_id"]) for r in report["records"]}
                     for split, report in reports.items()}
def labels(meta):
    transform = meta.get("transform") or {}
    scales = [transform.get("scale_x"), transform.get("scale_y")]
    resampling = None
    if all(isinstance(v, (int, float)) for v in scales):
        resampling = "mixed" if min(scales) < 1 < max(scales) else "upsampled" if max(scales) > 1 else "downsampled" if min(scales) < 1 else "unchanged"
    return (meta.get("category"), meta.get("actual_pitch_bin_x"), resampling)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for index, (ax, title) in enumerate(zip(axes, ("Crop category", "Cell pitch X", "Resampling"))):
    groups = {s: {} for s in splits_to_compare}
    for split in splits_to_compare:
        for record in reports[split]["records"]:
            if not record["positive"]:
                continue
            label = labels(metadata_by_split[split][record["sample_id"]])[index]
            if label is not None:
                groups[split].setdefault(str(label), []).append(record)
    keys = sorted(set(groups["val"]) | set(groups["test"]))
    for i, (split, color) in enumerate((("val", "#648bc5"), ("test", "#54a68b"))):
        for x, key in enumerate(keys):
            group = groups[split].get(key, [])
            position = x + (i-.5)*.36
            if group:
                correct = sum(bool(r.get("local_accepted") and r.get("local_correct") is True) for r in group)
                value = 100 * correct / len(group)
                ax.bar(position, value, .36, color=color)
                ax.text(position, min(value + 2, 99), f"n={len(group)}", ha="center", va="bottom", fontsize=7, rotation=90)
            else:
                ax.text(position, 2, "—", ha="center", fontsize=9)
    ax.set(title=title, ylabel="Correct local recovery (%)", ylim=(0, 110))
    ax.set_xticks(range(len(keys)), keys, rotation=35, ha="right", fontsize=8)
    if not keys: ax.text(.5, .5, "Unavailable", ha="center", transform=ax.transAxes)
from matplotlib.patches import Patch
fig.legend(handles=[Patch(color="#648bc5", label="Validation"), Patch(color="#54a68b", label="Test")], loc="upper center", ncol=2)
fig.tight_layout(rect=(0, 0, 1, .93))
plt.show()


## 8. Visualize predictions

Compare predicted phase and occupancy maps, recovered lattice and recognition outcomes with ground truth in representative examples and selected failures.


In [ ]:
DIAGNOSTIC_SPLIT = "val"
visualize_predictions(reports, metadata_by_split, saved_root, best_state, DEVICE,
                      EVALUATION_LIMIT, EVALUATION_SEED, DIAGNOSTIC_SPLIT)
